# 04. Selección de variables (RFE con XGBoost)

Selecciona los **200 genes** más discriminativos para el modelado SparkML (05). La separabilidad de las 18 clases (t-SNE) se documenta al final del EDA (03).


In [0]:
import subprocess
import sys

# Install packages with versions compatible with Databricks
# databricks-connect requires numpy<2 and pandas<3
packages = ["numpy>=1.25,<2", "pandas>=1.5,<3", "xgboost", "scikit-learn", "imbalanced-learn", "sdv", "openTSNE"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--quiet", pkg])

print("Packages installed successfully.")
dbutils.library.restartPython()


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow-skinny 2.11.4 requires protobuf<5,>=3.12.0, but you have protobuf 5.29.3 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.

Packages installed successfully.


In [0]:
%run ./00_configuracion


In [0]:
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.feature_selection import RFE
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    roc_auc_score, f1_score, average_precision_score
)
from xgboost import XGBClassifier
import joblib
import matplotlib.pyplot as plt

TOP_N_GENES    = 500
N_FEATURES_RFE = 200

print("Librerías cargadas")


Librerías cargadas


## 1. Carga y split (a nivel de paciente)

El split se calcula **antes** de mirar los genes, para que el prefiltro supervisado no use informacion de validation/test.


In [0]:
df_trusted = read_trusted(TRUSTED_LONG_PATH)
df_trusted.createOrReplaceTempView("trusted_long")
print("Registros:", df_trusted.count())

# Split a nivel de paciente
pats = df_trusted.select("patient_id", "cancer_type").dropDuplicates().toPandas()
try:
    tr, tmp = train_test_split(pats, test_size=0.30, random_state=SEED, stratify=pats["cancer_type"])
    val, te = train_test_split(tmp,  test_size=0.50, random_state=SEED, stratify=tmp["cancer_type"])
except ValueError:
    tr, tmp = train_test_split(pats, test_size=0.30, random_state=SEED)
    val, te = train_test_split(tmp,  test_size=0.50, random_state=SEED)

train_patients = set(tr["patient_id"]); val_patients = set(val["patient_id"]); test_patients = set(te["patient_id"])
print(f"Pacientes -> train {len(train_patients)} | val {len(val_patients)} | test {len(test_patients)}")
print("Solapamiento train-test:", len(train_patients & test_patients))

# Guardar el split de pacientes
split_map = pd.concat([
    tr[["patient_id", "cancer_type"]].assign(split="train"),
    val[["patient_id", "cancer_type"]].assign(split="validation"),
    te[["patient_id", "cancer_type"]].assign(split="test"),
], ignore_index=True)
save_spark_table(spark.createDataFrame(split_map), "refined_split_pacientes")
print("Split guardado en refined_split_pacientes:", len(split_map), "pacientes")


## 2. Prefiltro de genes más variables (solo con train)

Se rankean los genes por su variabilidad entre clases usando **únicamente las muestras de entrenamiento**. Así el conjunto de genes candidatos no se contamina con val/test. La tabla descriptiva `refined_eda_genes_mas_variables` de 03 (sobre todo el dataset) se mantiene solo para el EDA.


In [0]:
# Genes mas variables entre clases sobre muestras de entrenamiento
spark.createDataFrame(pd.DataFrame({"patient_id": sorted(train_patients)})) \
     .createOrReplaceTempView("train_patients_vw")

df_prefiltro_train = spark.sql(f"""
    WITH long_train AS (
        SELECT l.gene_id_base, l.gene_name, l.cancer_type, l.log2_tpm
        FROM trusted_long l
        JOIN train_patients_vw t ON l.patient_id = t.patient_id
    ),
    media_por_clase AS (
        SELECT gene_id_base, gene_name, cancer_type, AVG(log2_tpm) AS media_clase
        FROM long_train GROUP BY gene_id_base, gene_name, cancer_type
    ),
    variabilidad AS (
        SELECT gene_id_base, gene_name, STDDEV(media_clase) AS sd_entre_clases
        FROM media_por_clase GROUP BY gene_id_base, gene_name
    )
    SELECT gene_id_base, gene_name, ROUND(sd_entre_clases, 4) AS sd_entre_clases
    FROM variabilidad
    ORDER BY sd_entre_clases DESC
    LIMIT {TOP_N_GENES}
""")
save_spark_table(df_prefiltro_train, "refined_prefiltro_genes_train")

genes_top = [r["gene_id_base"] for r in df_prefiltro_train.select("gene_id_base").collect()]
print(f"Genes prefiltrados (solo train): {len(genes_top)}")


## 3. Matriz de expresión (genes prefiltrados)


In [0]:
pivot_cols = ["sample_id", "patient_id", "cancer_type"] if "patient_id" in df_trusted.columns else ["sample_id", "cancer_type"]

df_wide_spark = (
    df_trusted
    .filter(F.col("gene_id_base").isin(genes_top))
    .groupBy(*pivot_cols)
    .pivot("gene_id_base", genes_top)
    .agg(F.first("log2_tpm"))
    .fillna(0.0)
)

print("Convirtiendo a pandas...")
df_wide = df_wide_spark.toPandas()
FEATURE_COLS = [c for c in df_wide.columns if c not in ("sample_id", "patient_id", "cancer_type")]
df_wide[FEATURE_COLS] = df_wide[FEATURE_COLS].apply(pd.to_numeric, errors="coerce").fillna(0.0)
print(f"Shape: {df_wide.shape}")


## 4. Asignación train / val / test


In [0]:
# Asignacion de filas segun el split de pacientes
df_train = df_wide[df_wide["patient_id"].isin(train_patients)].copy()
df_val   = df_wide[df_wide["patient_id"].isin(val_patients)].copy()
df_test  = df_wide[df_wide["patient_id"].isin(test_patients)].copy()

le = LabelEncoder()
X_train = df_train[FEATURE_COLS].fillna(0)
X_val   = df_val[FEATURE_COLS].fillna(0)
X_test  = df_test[FEATURE_COLS].fillna(0)
y_train = le.fit_transform(df_train["cancer_type"])
y_val   = le.transform(df_val["cancer_type"])
y_test  = le.transform(df_test["cancer_type"])

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Clases: {list(le.classes_)}")
overlap = set(df_train["patient_id"]) & set(df_test["patient_id"])
print(f"Solapamiento train-test: {len(overlap)} ({'OK sin leakage' if len(overlap)==0 else 'LEAKAGE'})")


## 5. Selección de variables — RFE con XGBoost (500 → 200 genes)


In [0]:
n_clases = len(le.classes_)

estimador_rfe = XGBClassifier(
    n_estimators=50, max_depth=4,
    objective="multi:softprob", num_class=n_clases,
    eval_metric="mlogloss", random_state=SEED, n_jobs=-1, verbosity=0
)

rfe = RFE(estimator=estimador_rfe, n_features_to_select=N_FEATURES_RFE, step=0.1, verbose=1)
print(f"RFE: {X_train.shape[1]} → {N_FEATURES_RFE} genes...")
rfe.fit(X_train, y_train)

genes_rfe = [FEATURE_COLS[i] for i, s in enumerate(rfe.support_) if s]
print(f"Genes seleccionados: {len(genes_rfe)}")
print(genes_rfe[:10], "...")


## 6. Validación de las variables seleccionadas (XGBoost)


In [0]:
X_train_rfe = X_train[genes_rfe]
X_val_rfe   = X_val[genes_rfe]
X_test_rfe  = X_test[genes_rfe]

clf = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    objective="multi:softprob", num_class=n_clases,
    eval_metric="mlogloss", random_state=SEED, n_jobs=-1, verbosity=0
)
clf.fit(X_train_rfe, y_train, eval_set=[(X_val_rfe, y_val)], verbose=False)
print("Modelo entrenado")


## 7. Evaluación


In [0]:
resultados = []
for nombre, X_ev, y_ev in [("train", X_train_rfe, y_train), ("validation", X_val_rfe, y_val), ("test", X_test_rfe, y_test)]:
    y_pred  = clf.predict(X_ev)
    y_proba = clf.predict_proba(X_ev)
    try:
        roc = roc_auc_score(y_ev, y_proba, multi_class="ovr", average="macro")
    except:
        roc = float("nan")
    resultados.append({
        "split": nombre,
        "accuracy":          round(float(accuracy_score(y_ev, y_pred)), 4),
        "balanced_accuracy": round(float(balanced_accuracy_score(y_ev, y_pred)), 4),
        "f1_macro":          round(float(f1_score(y_ev, y_pred, average="macro", zero_division=0)), 4),
        "f1_weighted":       round(float(f1_score(y_ev, y_pred, average="weighted", zero_division=0)), 4),
        "roc_auc_macro":     round(roc, 4) if not np.isnan(roc) else None,
    })
    print(f"{nombre.upper():12s} acc={resultados[-1]['accuracy']:.4f}  f1_macro={resultados[-1]['f1_macro']:.4f}  ROC-AUC={resultados[-1]['roc_auc_macro']}")

df_metricas = pd.DataFrame(resultados)
display(df_metricas)

print("\nReporte detallado — Test:")
print(classification_report(y_test, clf.predict(X_test_rfe), target_names=le.classes_, zero_division=0))


## 8. Importancia de genes


In [0]:
importancias = (
    pd.DataFrame({"gene": genes_rfe, "importance": clf.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

plt.figure(figsize=(13, 5))
top = importancias.head(20)
plt.bar(top["gene"], top["importance"])
plt.title("Top 20 genes por importancia - XGBoost RFE")
plt.xlabel("Gen"); plt.ylabel("Importancia")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
guardar_figura("feature_importance_top20.png")
plt.show()

display(importancias.head(20))


## 9. Guardar variables seleccionadas y resultados


In [0]:
save_spark_table(spark.createDataFrame(importancias), "refined_rfe_feature_importances")
save_spark_table(spark.createDataFrame(df_metricas), "refined_metricas_xgboost_rfe")

joblib.dump(clf,       MODELS_PATH / "xgboost_rfe_multiclase.joblib")
joblib.dump(le,        MODELS_PATH / "label_encoder.joblib")
pd.DataFrame({"gene": genes_rfe}).to_csv(MODELS_PATH / "genes_rfe.csv", index=False)

print("Modelo guardado en:", MODELS_PATH)
print("Genes guardados:", MODELS_PATH / "genes_rfe.csv")
